## Урок 7. Дообучение моделей

### План

1. Классификация
    1. BERT fine-tuning
    1. GPT Prompt tuning
1. Генерация стихов с Mistral 7B
    1. Prompt tuning
    1. LoRA

In [2]:
!pip install  datasets --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.0 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

device = torch.device('cuda')

In [5]:
device

device(type='cuda')

# Классификация

В этом уроке мы, наконец, покончим с классификацией AG News.

Вспомним бейзлайны с предыдущих уроков:

| Метод | Точность |
| ----------- | ----------- |
| BoW + LogReg      | 0.909 |
| TF-IDF + LogReg   | __0.917__ |
| Усреднение Word2Vec   | 0.88 |
| TextCNN  | 0.898 |
| TextCNN + обучаемые эмбеддинги   | 0.912 |
| BERT пробинг   | 0.9 |

In [7]:
dataset = load_dataset("fancyzhx/ag_news", verification_mode='no_checks')
dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [6]:
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## BERT fine-tuning

Начнем с полного дообучения BERT. Для обучения модели мы будем будем использовать `transformers.Trainer`. Это очень удобный класс, в котором реализованы все циклы обучения.

In [11]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

dataset = dataset.map(lambda sample: tokenizer(sample['text']), batched=True)
# переименовываем колонку, потому что модель берет правильные ответы из `labels`
# при подсчете ошибки
dataset = dataset.rename_column('label', 'labels')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [16]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [17]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
training_args = TrainingArguments(
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=250,  # добавляем небольшой warmup (так как модель предобучена, он не очень нужен)
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,  # обучаем в пониженной точности для экономии ресурсов и ускорения
    num_train_epochs=1,
    logging_steps=100,  # логируем метрики раз в 100 шагов
    output_dir='outputs',
    report_to='wandb',
    run_name='bert_finetuning',
)

trainer = Trainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: theotheo46 (theotheo46-trs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,1.218400
200,0.434400
300,0.286000
400,0.240700
500,0.220900
600,0.229300
700,0.227600
800,0.215200
900,0.204600
1000,0.202300


TrainOutput(global_step=1875, training_loss=0.2704563278198242, metrics={'train_runtime': 734.9918, 'train_samples_per_second': 163.267, 'train_steps_per_second': 2.551, 'total_flos': 8401418416355328.0, 'train_loss': 0.2704563278198242, 'epoch': 1.0})

In [19]:
trainer.evaluate()

{'eval_loss': 0.17471030354499817,
 'eval_accuracy': 0.9410526315789474,
 'eval_runtime': 12.3096,
 'eval_samples_per_second': 617.405,
 'eval_steps_per_second': 9.667,
 'epoch': 1.0}

Получаем точность на 2% выше всех нетрансформерных моделей!

Несмотря на этот успех, важно не забывать, что такое улучшение можно получить только для сложных задач, с которыми простые методы плохо справляются.

## Prompt tuning

PEFT метод, обучающий промпт для языковой модели, а так же голову. Может применяться как для любой классификации текста, так и для генерации.

<img src="https://i.ibb.co/54bv1Gt/prompt-tuning-1.png" alt="drawing" width="450"/>

In [7]:
from peft import TaskType, PromptTuningConfig, PromptTuningInit, get_peft_model

In [8]:
# добавляем паддинги слева, потому что gpt будет возвращать выход последнего токена в батче
tokenizer = AutoTokenizer.from_pretrained("gpt2", padding_side='left')
# по умолчанию в gpt нет токенов паддинга
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [9]:
dataset = load_dataset("fancyzhx/ag_news")
dataset = dataset.map(lambda sample: tokenizer(sample['text'], return_token_type_ids=False), batched=True)
dataset = dataset.rename_column('label', 'labels')
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=4)
# токен паддинга нужно указать отдельно из-за особенностей реализации gpt
model.config.pad_token_id = tokenizer.pad_token_id

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
prompt_size = 8
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # нужно для правильного подсчета ошибки
    num_virtual_tokens=prompt_size,  # число обучаемых токенов промпта
)

pt_model = get_peft_model(model, peft_config).to(device)
pt_model.print_trainable_parameters()

trainable params: 6,144 || all params: 124,449,024 || trainable%: 0.0049


In [12]:
training_args = TrainingArguments(
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=3e-2,  # выставляем очень большой lr!
    weight_decay=0.01,
    fp16=True,
    num_train_epochs=4,
    logging_steps=100,
    output_dir='outputs',
    report_to='wandb',
    run_name='gpt_pt',
)

trainer = Trainer(
    model=pt_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

<ipython-input-12-2a14bffd9e2f>:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: theotheo46 (theotheo46-trs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPT2ForSequenceClassification will not detect padding tokens in `inputs_embeds`. Results may be unexpected if using padding tokens in conjunction with `inputs_embeds.`


Step,Training Loss
100,1.759800
200,1.481100
300,1.445700
400,1.469300
500,1.428500
600,1.414600
700,1.405800
800,1.398900
900,1.394900
1000,1.393500


TrainOutput(global_step=7500, training_loss=0.6663347752888997, metrics={'train_runtime': 3049.0143, 'train_samples_per_second': 157.428, 'train_steps_per_second': 2.46, 'total_flos': 3.223728485616845e+16, 'train_loss': 0.6663347752888997, 'epoch': 4.0})

In [13]:
trainer.evaluate()

{'eval_loss': 0.3672696053981781,
 'eval_accuracy': 0.8932894736842105,
 'eval_runtime': 19.2511,
 'eval_samples_per_second': 394.783,
 'eval_steps_per_second': 6.181,
 'epoch': 4.0}

Финальное качество на 2% меньше, чем у полного дообучения BERT. Однако оно превосходит качество пробинга BERT. Можно сделать вывод о том, что prompt tuning действительно работает. Попробуем теперь применить его для более интересной задачи.

### Резюме

1. Узнали, как дообучать модели с помощью класса `Trainer`.
1. Выяснили, что не небольшой задаче классификации fine-tuning работает лучше, чем prompt tuning.
1. Для дообучения моделей по умолчанию всегда лучше использовать fine-tuning. Его очень редко удается обогнать.

## Генерация стихов

Представим, что мы хотим дообучить огромную модель (Mistral-7B) генерировать стихи на разные темы. Так как модель огромная, ей точно хватает знаний для этого, однако формат выхода нас может не устраивать.

In [1]:
import torch
import wandb

device = torch.device('cuda')

In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# В этом ноутбуке модель работает на V100 с 32 гб памяти
# Если у вас не хватает памяти, то можно взять модель поменьше, например, microsoft/phi-1_5
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-1_5", device_map='cuda', torch_dtype=torch.float16
)

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
print("GPU memory allocated: %fGB" % (torch.cuda.memory_allocated() / 2**30))
print("Max GPU memory: %fGB" % (torch.cuda.get_device_properties('cuda').total_memory / 2**30))

GPU memory allocated: 13.988792GB
Max GPU memory: 31.739380GB


Так как модель занимает 14 гб памяти, дообучать ее целиком мы точно не можем. При обратном проходе нужно хранить градиенты для каждого веса, а так же гессианы, если мы используем Adam.

Для начала проверим, может ли предобученная модель генерировать стихи в нужном формате без дообучения.

In [ ]:
prompt = 'Human: Can you write me a poem about faults and love? Assistant:'
inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    top_p=0.95,
    do_sample=True,
    return_dict_in_generate=False,
).cpu()[0]

print(tokenizer.decode(outputs))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> Human: Can you write me a poem about faults and love? Assistant: I thought you didn’t believe in true love. Human: I don’t know. I want someone to tell me that it’s real. Someone I can actually kiss. Assistant: I think you have some unresolved emotions from your last relationship. […]

Here’s a sample chapter from my upcoming book “The Writer of Dreams and Other Tales of Love, Sex, and Death” which will be out this December!

I wanted to write a book about love


Явно не может. Что и логично, так как ее задача – максимально правдоподобно генерировать продолжение.

Загрузим небольшой датасет стихов, на который будем дообучать модель.

In [ ]:
from datasets import load_dataset, load_from_disk

dataset = load_dataset("iamketan25/poem-instructions-dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen'],
        num_rows: 1764
    })
    test: Dataset({
        features: ['prompt', 'chosen'],
        num_rows: 331
    })
})

In [ ]:
dataset['train'][0]

{'prompt': 'Human: Can you write me a poem about faults and love?\nAssistant:',
 'chosen': " Sure, here's a poem about faults and love:\n  They came to tell your faults to me,\nThey named them over one by one;\nI laughed aloud when they were done,\nI knew them all so well before, \nOh, they were blind, too blind to see\nYour faults had made me love you more."}

Каждый стих содержит запрос с указанием тематики и результат генерации. Заметьте, что это seq2seq задача, однако ничего нам не мешает дообучать на нее GPT-like модель, подавая запрос в качестве начала последовательности.

Предобработаем датасет, чтобы он имел нужный для обучения формат.

In [ ]:
from copy import deepcopy

def preprocess(samples):
    texts = [prompt + ans for prompt, ans in zip(samples['prompt'], samples['chosen'])]
    texts_tokenized = tokenizer(
        texts,
        max_length=196,  # ограничиваем максимальный размер текста
        truncation=True,
        return_token_type_ids=False
    )
    prompts_tokenized = tokenizer(
        samples['prompt'],
        return_token_type_ids=False
    )

    labels = deepcopy(texts_tokenized['input_ids'])

    # для обучения seq2seq задаче мы не должны штрафовать модель за предсказания промпта,
    # поэтому заменим лейблы для него значением -100
    for i in range(len(labels)):
        prompt_len = len(prompts_tokenized['input_ids'][i])
        labels[i][:prompt_len] = [-100] * prompt_len

    return {
        'input_ids': texts_tokenized['input_ids'],
        'attention_mask': texts_tokenized['attention_mask'],
        'labels': labels
    }

In [ ]:
dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
# удаляем все тексты, в которых промпт слишком длинный
dataset = dataset.filter(lambda x: not all(x['labels'] == -100))

Filter: 100%|██████████| 331/331 [00:05<00:00, 61.93 examples/s]


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1449
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 267
    })
})

Сохраним обработанный датасет, чтобы не обрабатывать его заново в случае чего.

In [ ]:
dataset.save_to_disk("poem_instructions")

Saving the dataset (1/1 shards): 100%|██████████| 267/267 [00:00<00:00, 6209.96 examples/s]


In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("poem_instructions")

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Будем оценивать качество модели с помощью точности предсказания токенов. Для этого переопределим функцию подсчета метрики.

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    # удаляем предсказания для обучаемого промпта в случае prompt tuning
    predictions = predictions[:, predictions.shape[1] - labels.shape[1]:]

    # сдвигаем предсказания, чтобы для каждого токена
    # считать правильность предсказания следующего
    shifted_labels = labels[:, 1:]
    shifted_predictions = predictions[:, :-1]

    mask = shifted_labels != -100

    return accuracy.compute(predictions=shifted_predictions[mask], references=shifted_labels[mask])

## Prompt Tuning

In [24]:
from peft import TaskType, PromptTuningConfig, PromptTuningInit, get_peft_model

In [25]:
prompt_size = 20
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,  # выбираем задачу генерации текста
    num_virtual_tokens=prompt_size,
)

pt_model = get_peft_model(model, peft_config).to(device)
pt_model.print_trainable_parameters()

trainable params: 15,360 || all params: 124,458,240 || trainable%: 0.0123


Определим функцию для обработки батча с паддингами для всех входных тензоров.

In [26]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = [sample['input_ids'] for sample in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)

    attention_mask = [sample['attention_mask'] for sample in batch]
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)

    labels = [sample['labels'] for sample in batch]
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    # берем единичный батч, потому что больше не влезает
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    # аккумулируем градиенты за несколько шагов, чтобы увеличить батч
    gradient_accumulation_steps=4,
    learning_rate=3e-2,  # опять выставляем большой lr
    weight_decay=0.01,
    fp16=True,
    num_train_epochs=8,
    logging_steps=100,
    output_dir='outputs',
    report_to='wandb',
    run_name='mistral_pt',
)

trainer = Trainer(
    model=pt_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
    tokenizer=tokenizer,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Сперва посчитаем точность предсказания модели до дообучения промпта.

In [ ]:
trainer.evaluate()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ashabalin. Use `wandb login --relogin` to force relogin


{'eval_loss': 3.263199806213379,
 'eval_accuracy': 0.39523786571552516,
 'eval_runtime': 56.6948,
 'eval_samples_per_second': 4.709,
 'eval_steps_per_second': 4.709}

In [ ]:
trainer.train()

Step,Training Loss
100,2.353100
200,2.108900
300,2.100400
400,2.069700
500,2.066300
600,2.012200
700,2.022700
800,2.041800
900,2.023000
1000,2.016900


TrainOutput(global_step=2896, training_loss=2.0184080692944604, metrics={'train_runtime': 2095.2887, 'train_samples_per_second': 5.532, 'train_steps_per_second': 1.382, 'total_flos': 9.20729226632233e+16, 'train_loss': 2.0184080692944604, 'epoch': 7.99})

In [ ]:
# pt_model = pt_model.from_pretrained(model, 'pt_model')

Теперь оценим точность после дообучения. Видим, что она поднялась на 20%, а значит, обучение промпта помогло.

In [ ]:
trainer.evaluate()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ashabalin. Use `wandb login --relogin` to force relogin


{'eval_loss': 2.0548131465911865,
 'eval_accuracy': 0.5742757989106859,
 'eval_runtime': 56.5656,
 'eval_samples_per_second': 4.72,
 'eval_steps_per_second': 4.72}

In [ ]:
pt_model.save_pretrained('pt_model')

Посмотрим теперь, как изменились выходы модели.

In [ ]:
prompt = 'Human: Can you write me a poem about lemons and cherries? Assistant:'
inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

In [ ]:
outputs = pt_model.generate(
    **inputs,
    max_new_tokens=100,
    top_p=0.95,
    do_sample=True,
    return_dict_in_generate=False,
).cpu()[0]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
print(tokenizer.decode(outputs))

<s> Human: Can you write me a poem about lemons and cherries? Assistant: Sure, here's a poem about lemons and cherries:

                And I
Would have the sweetest, and then the fairest,
And wear them both, till Hymen's girdle day,
And marry with my lovely cherries,
While lemon-sweet lemon-fairly doth say,
I am as sweet, and so much fairer
Than cherries!  Cherry, answer this chiding,



Все еще видны явные проблемы с рифмой, но теперь модель всегда генерирует стих, а не что-то еще.

## LoRA

Метод PEFT, который обучает сдвиг для матриц $W_q$ и $W_v$.

<img src="https://i.ibb.co/Sxq33GY/lora-1.png" alt="drawing" width="400"/>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1", device_map='cuda', torch_dtype=torch.float16
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.47s/it]


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_dropout=0.1
)

lora_model = get_peft_model(model, peft_config).to(device)
lora_model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


Обучаем гораздо больше параметров, чем в prompt tuning. Зато получим возможность корректировать знания модели.

In [ ]:
peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='mistralai/Mistral-7B-v0.1', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, lora_alpha=8, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,  # добавляем чекпойнты для экономии памяти
    learning_rate=3e-4,  # более менее стандартный lr
    weight_decay=0.01,
    fp16=True,
    num_train_epochs=8,  # LoRA сходится в разы дольше
    logging_steps=100,
    output_dir='outputs',
    report_to='wandb',
    run_name='mistral_lora',
)

trainer = Trainer(
    model=lora_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
    tokenizer=tokenizer,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:
trainer.train()

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the futu

Step,Training Loss
100,2.137400
200,2.028200
300,2.031000
400,1.988100
500,1.916700
600,1.862400
700,1.876000
800,1.788000
900,1.728900
1000,1.747500


/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/peft/utils/other.py:611: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /mistralai/Mistral-7B-v0.1/resolve/main/config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x2ac5753d0e20>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: 15a37d41-9d4a-49

TrainOutput(global_step=2896, training_loss=1.5322605338544477, metrics={'train_runtime': 4226.0216, 'train_samples_per_second': 2.743, 'train_steps_per_second': 0.685, 'total_flos': 9.211704975281357e+16, 'train_loss': 1.5322605338544477, 'epoch': 7.99})

In [ ]:
lora_model.save_pretrained('lora_model')

По изменению ошибки видим, что модель обучилась не до сходимости. Однако тем не менее остановим ее тут и замеряем точность предсказаний.

In [ ]:
trainer.evaluate()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ashabalin. Use `wandb login --relogin` to force relogin


{'eval_loss': 1.0405954122543335,
 'eval_accuracy': 0.7587121029546441,
 'eval_runtime': 51.995,
 'eval_samples_per_second': 5.135,
 'eval_steps_per_second': 5.135}

Видим, что результат получился значительно лучше, чем у prompt tuning. К сожалению, убедиться в этом, смотря на стихи, мы не сможем, потому что рифма лучше не стала. Поэтому давайте просто посмотрим на стих про обезьяну.

In [ ]:
prompt = 'Human: Can you write me a poem about monkey in a zoo? Assistant:'
inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

In [ ]:
outputs = lora_model.generate(
    **inputs,
    max_new_tokens=100,
    top_p=0.95,
    do_sample=True,
    return_dict_in_generate=False,
).cpu()[0]

print(tokenizer.decode(outputs))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> Human: Can you write me a poem about monkey in a zoo? Assistant: Sure, here's a poem about monkey in a zoo:
    I'm so little
You're so big
What a big hand you have
That's a monkey in a zoo
He lives there
He doesn't belong to anybody
Nobody takes care of him
His name is Monkey
Look how high he can climb
He eats bananas and bread and nuts
He's as brown as a nut
He


### Резюме

1. Научились дообучать модели на задачу классификации.
1. Разобрались с двумя PEFT методами: Prompt tuning и LoRA.
1. Узнали, как обучать LLM с их помощью и какие подводные камни встречаются на этом пути.